# PySpark Config Options

Explore Spark configuration at runtime — list all options, set/get values
dynamically, and validate settings with `SparkConf`.

In [ ]:
import os

from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("config-options-notebook")
         .master(os.environ.get("SPARK_MASTER", "local[*]"))
         .config("spark.sql.adaptive.enabled", "true")
         .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
         .config("spark.sql.shuffle.partitions", "4")
         .config("spark.ui.enabled", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)

## List All SparkContext Config

In [ ]:
for key, value in sorted(spark.sparkContext.getConf().getAll()):
    print(f"  {key} = {value}")

## List All Spark SQL Config

`SET -v` returns every SQL-level config with its description.

In [ ]:
sql_configs = spark.sql("SET -v")
sql_configs.show(20, truncate=False)

## Get / Set Config at Runtime

In [ ]:
# Read current value
print("shuffle.partitions =", spark.conf.get("spark.sql.shuffle.partitions"))

# Change at runtime
spark.conf.set("spark.sql.shuffle.partitions", "8")
print("shuffle.partitions =", spark.conf.get("spark.sql.shuffle.partitions"))

# Reset
spark.conf.unset("spark.sql.shuffle.partitions")
print("shuffle.partitions =", spark.conf.get("spark.sql.shuffle.partitions"))

## Validate with SparkConf

In [ ]:
from pyspark import SparkConf

conf = spark.sparkContext.getConf()
print("app.name         =", conf.get("spark.app.name"))
print("master           =", conf.get("spark.master"))
print("adaptive.enabled =", conf.get("spark.sql.adaptive.enabled"))

# get() with default fallback
print("speculation      =", conf.get("spark.speculation", "false"))

## Immutable vs Mutable Config

Some config keys (e.g. `spark.master`) cannot be changed after session creation.

In [ ]:
try:
    spark.conf.set("spark.master", "yarn")
except Exception as e:
    print(f"Expected error: {e}")

In [ ]:
spark.stop()